# Getting Started with KinDER

This notebook walks through the basics of the KinDER benchmark:
discovering available environments, creating one, taking actions,
and rendering a video.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import kinder

## Discovering Available Environments

KinDER organizes environments into categories. Let's register them all
and see what's available.

In [ ]:
kinder.register_all_environments()

categories = kinder.get_env_categories()
for category, env_classes in sorted(categories.items()):
    print(f"{category}: {len(env_classes)} environment classes")
    for cls in env_classes:
        variants = kinder.get_env_variants(cls)
        print(f"  {cls} ({len(variants)} variants)")

## Creating an Environment

We'll use `Obstruction2D-o3-v0`, a Kinematic2D environment where a robot
must place a block onto a target surface while navigating around obstacles.

In [ ]:
env = kinder.make("kinder/Obstruction2D-o3-v0", render_mode="rgb_array")
obs, info = env.reset(seed=42)

frame = env.render()
plt.imshow(frame)
plt.axis("off")
plt.title("Obstruction2D-o3-v0")
plt.show()

## Exploring the Observation and Action Spaces

KinDER environments follow the Gymnasium API. Observations and actions
are continuous-valued numpy arrays.

In [ ]:
print("Observation shape:", env.observation_space.shape)
print("Action shape:     ", env.action_space.shape)
print()

action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)
print("Sample action:", np.round(action, 3))
print("Reward:       ", reward)
print("Terminated:   ", terminated)

## Rendering a Video

Let's collect frames from random actions and display them as an animated GIF.

In [ ]:
from io import BytesIO

from IPython.display import Image
from PIL import Image as PILImage

obs, info = env.reset(seed=0)
frames = [env.render()]
for _ in range(50):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    frames.append(env.render())
    if terminated or truncated:
        break

pil_frames = [PILImage.fromarray(f) for f in frames]
buf = BytesIO()
pil_frames[0].save(
    buf,
    format="GIF",
    save_all=True,
    append_images=pil_frames[1:],
    duration=100,
    loop=0,
)
Image(data=buf.getvalue(), format="gif")

## Cleanup

In [ ]:
env.close()